# Human Action Recognition: Searching for a Good Fit with CLIP ViT-B/16

**Goal:** training curves with a **good fit**: training and validation accuracy rise, flatten and stay close together
(a small gap), with validation accuracy as high as possible. Neither underfitting nor overfitting.

**Approach.** Train **ten variants of CLIP ViT-B/16**. Variant A is the best run so far; every other variant changes
**one thing**, so its effect on the curve is clear. Variant J combines several regularizers.

| Lever | What it does to the curves |
|---|---|
| **Freezing blocks** | Fewer trainable weights → less memorising, smaller gap. Too many frozen → underfitting. |
| **Learning rate** | Lower → slower, smoother rise over more epochs. |
| **Dropout** | Randomly switches off features before the classifier → slows memorising. |
| **Drop path** | Randomly skips whole transformer blocks while training → a stronger regularizer for transformers. |
| **Strong augmentation** | Bigger random changes to each training photo (TrivialAugment, random erasing) → harder to memorise. |
| **Mixup / CutMix** | Blends pairs of photos and their labels → one of the strongest fixes for overfitting. |
| **Stopping on validation loss** | Loss rises before accuracy falls, so it can stop before memorising starts. |

**Two ways to measure training accuracy.** The first notebook (03) measured it on *augmented* photos with dropout on,
while training. Later notebooks measured it on *clean* photos after each epoch. Clean is stricter: it shows memorising
and gives a bigger gap. This notebook records **both**, so the curves can be compared fairly.

**Choosing a variant (written in code):** keep variants whose gap (clean training − validation) at the kept epoch is
**5 points or less**, then take the highest **validation** accuracy. The test set is reported, never used to choose.

> **On Kaggle:** **Accelerator → GPU T4 x2**, **Internet → On**, attach the *Human Action Recognition (HAR) Dataset*,
> then **Save Version → Save & Run All**. Ten variants run two at a time, one per GPU: about **2–3 hours**.
> Finished variants are saved; if the session stops, running again skips them. To run fewer, delete lines from `VARIANTS`.

# Part 0 — Settings
Variant A uses `DEFAULTS`. Each other variant lists only what it changes.

In [ ]:
EPOCHS = 25            # maximum epochs for every variant; early stopping usually ends sooner
PATIENCE = 3           # early stopping: stop after 3 epochs without improvement
EMA_DECAY = 0.998      # weight averaging, the same for every variant (keeps curves smooth)

DEFAULTS = dict(
    freeze_blocks=6,       # lower transformer blocks kept frozen (0 = train all 12, 12 = train only the classifier)
    lr=5e-5,               # learning rate of the top block (each lower block x0.75, the classifier x5)
    dropout=0.05,          # dropout before the classifier
    drop_path=0.1,         # drop path (stochastic depth) inside the transformer
    weight_decay=0.05,     # AdamW weight decay
    label_smoothing=0.1,   # softens the targets
    augment="standard",    # "standard" (as in notebook 03) or "strong"
    mixup=False,           # mixup / cutmix on half of the batches
    monitor="val_acc",     # what early stopping watches: "val_acc" or "val_loss"
)
VARIANTS = {
    "A: baseline (freeze 6)": {},
    "B: freeze 10": {"freeze_blocks": 10},
    "C: freeze all (classifier only)": {"freeze_blocks": 12},
    "D: learning rate 2e-5": {"lr": 2e-5},
    "E: dropout 0.3": {"dropout": 0.3},
    "F: drop path 0.3": {"drop_path": 0.3},
    "G: strong augmentation": {"augment": "strong"},
    "H: mixup / cutmix": {"mixup": True},
    "I: stop on validation loss": {"monitor": "val_loss"},
    "J: combined": {"freeze_blocks": 8, "lr": 3e-5, "dropout": 0.2, "drop_path": 0.2, "augment": "strong", "mixup": True},
}

import pandas as pd
settings_table = pd.DataFrame([{"variant": n, **{**DEFAULTS, **o}} for n, o in VARIANTS.items()]).set_index("variant")
settings_table

# Part 1 — Data

## 1.1 Import libraries
This notebook uses **PyTorch** (a deep-learning framework, like Keras/TensorFlow) and **timm**
("PyTorch Image Models"), a library with hundreds of pretrained image networks, including CLIP.

In [ ]:
import os, gc, glob, copy, math, time, random, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import timm
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", message="Detected call of")   # harmless scheduler notice with mixed precision

SEED = 42
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| timm:", timm.__version__)
print("Device: ", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU - turn on the GPU accelerator in Settings!")

## 1.2 Find the dataset

In [ ]:
csv_matches = glob.glob("/kaggle/input/**/Training_set.csv", recursive=True)
assert csv_matches, "Dataset not found - click 'Add Input' and attach the Human Action Recognition dataset."
DATA_DIR = os.path.dirname(csv_matches[0])
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")
OUTPUT_DIR = "/kaggle/working"
print("Data folder:", DATA_DIR)

## 1.3 Load the labels
Each action gets a number (`calling → 0`, `clapping → 1`, …), because networks work with numbers.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "Training_set.csv"))
class_names = sorted(df["label"].unique())
label_to_id = {name: i for i, name in enumerate(class_names)}
df["label_id"] = df["label"].map(label_to_id)
NUM_CLASSES = len(class_names)

print(f"{len(df)} labeled images, {NUM_CLASSES} actions")
print(df["label"].value_counts().sort_index().to_string())

## 1.4 Example images

In [ ]:
sample = df.sample(8, random_state=1)
plt.figure(figsize=(14, 6))
for i, (_, row) in enumerate(sample.iterrows()):
    plt.subplot(2, 4, i + 1)
    plt.imshow(Image.open(os.path.join(TRAIN_IMG_DIR, row["filename"])))
    plt.title(row["label"])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 1.5 Find near-duplicate photos (prevent data leakage)
Some photos in the dataset are near-copies of each other (the same picture saved twice, resized or recompressed).
If one copy lands in the **training** set and the other in the **test** set, the model can simply *remember*
the photo, and the test accuracy looks better than it really is. This is called **data leakage**.

To prevent it we give every photo a **fingerprint** with *difference hashing (dHash)*:
1. Shrink the photo to 9 × 8 pixels in grayscale.
2. For each pixel, record whether it is brighter than its right-hand neighbour → **64 yes/no bits**.

Photos whose fingerprints differ in **4 bits or fewer** are put in the same **group**, and a group is never
split across train, validation and test.

The fingerprint is deliberately **cautious**. Besides real copies, it also matches some *different* photos that
share a large plain background (white studio shots, black borders), because plain areas give identical bits.
Keeping two different photos on the same side of the split doesn't leak anything, so that's harmless.
To see how many matches are real copies, we also compare small 32 × 32 colour thumbnails pixel by pixel.

In [ ]:
def dhash(path):
    img = Image.open(path).convert("L").resize((9, 8), Image.BILINEAR)
    px = np.asarray(img, dtype=np.int16)
    return (px[:, 1:] > px[:, :-1]).flatten()          # 8 rows x 8 comparisons = 64 bits

paths = [os.path.join(TRAIN_IMG_DIR, f) for f in df["filename"]]
with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    bits = np.array(list(pool.map(dhash, paths)), dtype=np.float32)   # shape: (images, 64)

# Hamming distance = number of bits that differ between two fingerprints
pair_a, pair_b = [], []
for start in range(0, len(bits), 1024):
    chunk = bits[start:start + 1024]
    agree = chunk @ bits.T + (1 - chunk) @ (1 - bits).T
    a, b = np.nonzero(64 - agree <= 4)
    a = a + start
    keep = b > a                                        # count each pair once, skip self-matches
    pair_a.append(a[keep])
    pair_b.append(b[keep])
pair_a, pair_b = np.concatenate(pair_a), np.concatenate(pair_b)

n = len(df)
_, groups = connected_components(coo_matrix((np.ones(len(pair_a)), (pair_a, pair_b)), shape=(n, n)), directed=False)
df["group"] = groups

# How alike are the matched pairs really? Average pixel difference of 32 x 32 colour thumbnails (0-255 scale)
def thumbnail(path):
    return np.asarray(Image.open(path).convert("RGB").resize((32, 32), Image.BILINEAR), dtype=np.float32)
pixel_diff = np.array([np.abs(thumbnail(paths[a]) - thumbnail(paths[b])).mean() for a, b in zip(pair_a, pair_b)])
is_copy = pixel_diff < 10
same_label = df["label"].values[pair_a] == df["label"].values[pair_b]

print(f"Photo pairs with matching fingerprints: {len(pair_a)}")
if len(pair_a):
    print(f"  clearly copies (thumbnails differ by less than 10 out of 255): {is_copy.sum()}"
          f" - same label in {same_label[is_copy].mean() if is_copy.any() else 0:.0%} of them")
    print(f"  other matches (near-copies such as crops, and look-alikes):      {(~is_copy).sum()}")
print(f"Groups: {groups.max() + 1} groups for {n} images")

**Examples.** The first three pairs are copies of the same photo; the differences are watermarks or image quality.
The last pair is a *look-alike*: two different photos that the fingerprint matched because of their plain backgrounds.

In [ ]:
copies = np.where(is_copy)[0]
copies = copies[np.argsort(-pixel_diff[copies])][:3]      # copies with small visible differences
lookalikes = np.where(~is_copy)[0]
lookalikes = lookalikes[np.argsort(pixel_diff[lookalikes])][len(lookalikes) // 2:][:1]   # a typical look-alike
rows = [(k, "copy") for k in copies] + [(k, "look-alike") for k in lookalikes]
if rows:
    plt.figure(figsize=(8, 3.2 * len(rows)))
    for r, (k, kind) in enumerate(rows):
        for j, idx in enumerate((pair_a[k], pair_b[k])):
            plt.subplot(len(rows), 2, 2 * r + j + 1)
            plt.imshow(Image.open(paths[idx]))
            title = f"{df['filename'][idx]} · {df['label'][idx]}"
            if j == 0:
                title = f"{kind.upper()} (pixel difference {pixel_diff[k]:.1f})\n" + title
            plt.title(title, fontsize=9)
            plt.axis("off")
    plt.tight_layout()
    plt.show()

## 1.6 Split into training, validation and test sets
- **Training (80%)** – the model learns from these photos.
- **Validation (10%)** – checked after every epoch to pick the best version of the model.
- **Test (10%)** – used **only once at the very end** to report the final accuracy.

Because the validation set is used to *choose* the best epoch, its score is slightly optimistic.
The untouched **test set** gives an honest final number. *Stratified* keeps the same mix of actions in every set;
*group* keeps near-duplicates together.

(The dataset's own `test` folder has no labels, so it can't measure accuracy. Part 6 explains what we use it for.)

In [ ]:
y_all = df["label_id"].values
all_idx = np.arange(len(df))

split_test = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(split_test.split(all_idx, y_all, groups))

split_val = StratifiedGroupKFold(n_splits=9, shuffle=True, random_state=SEED)
tr, va = next(split_val.split(trainval_idx, y_all[trainval_idx], groups[trainval_idx]))
train_idx, val_idx = trainval_idx[tr], trainval_idx[va]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

assert not (set(train_df.group) & set(val_df.group) or set(train_df.group) & set(test_df.group) or set(val_df.group) & set(test_df.group))
print(f"Training:   {len(train_df):>5} images")
print(f"Validation: {len(val_df):>5} images")
print(f"Test:       {len(test_df):>5} images")
print("No near-duplicate group is shared between the sets.")

# Part 2 — Preprocessing

## 2.1 Resize, normalize and augment
**Every image (validation and test):**
- **Resize** to 224 × 224 pixels — the input size CLIP ViT-B/16 was trained on.
- **Normalize** each colour channel with CLIP's own mean and standard deviation, so the inputs look like
  what the network saw during pretraining.

**Training images only — data augmentation** (a new random version every epoch):
- **Random resized crop** — zoom into 40–100% of the photo, so the model sees people at different sizes and positions.
- **Horizontal flip** — a mirrored person is doing the same action.
- **Small rotation** (±10°) and **brightness / contrast** changes (±20%).

Augmentation makes the training set look bigger and more varied, which reduces **overfitting**.

In [ ]:
IMG_SIZE = 224
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10, interpolation=InterpolationMode.BILINEAR),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])

def to_display(t):
    # Undo the normalization so a tensor can be shown as a picture
    mean, std = torch.tensor(CLIP_MEAN).view(3, 1, 1), torch.tensor(CLIP_STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

example = Image.open(os.path.join(TRAIN_IMG_DIR, train_df["filename"][0])).convert("RGB")
plt.figure(figsize=(14, 3.5))
plt.subplot(1, 6, 1); plt.imshow(to_display(eval_tf(example))); plt.title("original"); plt.axis("off")
for i in range(5):
    plt.subplot(1, 6, i + 2); plt.imshow(to_display(train_tf(example))); plt.title(f"augmented {i + 1}"); plt.axis("off")
plt.suptitle(f"Action: {train_df['label'][0]}")
plt.tight_layout()
plt.show()

## 2.2 Data loaders
A **Dataset** says how to load one image and its label; a **DataLoader** groups images into **batches of 32**
and loads them in parallel on the CPU while the GPU trains.

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count() or 2

class HARDataset(Dataset):
    def __init__(self, frame, img_dir, transform, labeled=True):
        self.files = frame["filename"].tolist()
        self.labels = frame["label_id"].tolist() if labeled else [-1] * len(frame)
        self.img_dir, self.transform = img_dir, transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(os.path.join(self.img_dir, self.files[i])).convert("RGB")
        return self.transform(img), self.labels[i]

def make_loader(frame, img_dir, transform, train, labeled=True):
    return DataLoader(HARDataset(frame, img_dir, transform, labeled),
                      batch_size=BATCH_SIZE if train else 128, shuffle=train, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
                      persistent_workers=NUM_WORKERS > 0)

train_loader = make_loader(train_df, TRAIN_IMG_DIR, train_tf, train=True)
val_loader = make_loader(val_df, TRAIN_IMG_DIR, eval_tf, train=False)
test_loader = make_loader(test_df, TRAIN_IMG_DIR, eval_tf, train=False)

x, y = next(iter(train_loader))
print("Batch of images:", tuple(x.shape), "-> 32 images, 3 colours, 224 x 224 pixels")
print("Batch of labels:", tuple(y.shape))
print("Training steps per epoch:", len(train_loader))

# Part 3 — The Training Code
## 3.1 Save the split and the settings
The training runs in separate processes (one per GPU), so the split from Part 1 and the settings from Part 0 are saved to
files they all read. Every variant uses exactly the same training, validation and test photos.

In [ ]:
import json, subprocess, sys
MODEL_NAME = "vit_base_patch16_clip_224.openai"
PRETRAINED = True
TRAIN_EVAL_N = 2000    # training photos re-measured on clean images after every epoch

WORK = os.path.join(OUTPUT_DIR, "search")
os.makedirs(WORK, exist_ok=True)
train_eval_df = train_df.sample(min(TRAIN_EVAL_N, len(train_df)), random_state=SEED).reset_index(drop=True)
for part, frame in (("train", train_df), ("val", val_df), ("test", test_df), ("train_eval", train_eval_df)):
    frame[["filename", "label_id"]].to_csv(os.path.join(WORK, f"{part}.csv"), index=False)

n_gpu = torch.cuda.device_count()
n_procs = 2 if n_gpu >= 2 else 1
config = dict(img_dir=TRAIN_IMG_DIR, num_classes=NUM_CLASSES, epochs=EPOCHS, patience=PATIENCE, ema_decay=EMA_DECAY,
              seed=SEED, model_name=MODEL_NAME, pretrained=PRETRAINED, img_size=IMG_SIZE, batch_size=BATCH_SIZE,
              mean=list(CLIP_MEAN), std=list(CLIP_STD), workers=max(1, (os.cpu_count() or 2) // n_procs),
              defaults=DEFAULTS, variants=VARIANTS, names=list(VARIANTS))
with open(os.path.join(WORK, "config.json"), "w") as f:
    json.dump(config, f, indent=1)
print(f"Saved the split ({len(train_df)} / {len(val_df)} / {len(test_df)} photos) and {len(VARIANTS)} variants to {WORK}")
print(f"GPUs: {n_gpu} -> {n_procs} training process(es) in parallel")

## 3.2 The training script
This is the code every variant runs. For each variant it:
1. Loads CLIP ViT-B/16 with a new 15-neuron classifier, freezes the lowest `freeze_blocks` blocks.
2. Trains with AdamW, layer-wise learning rates (each lower block ×0.75, classifier ×5), 1 warm-up epoch then cosine decay,
   label smoothing, gradient clipping, mixed precision, and (if set) strong augmentation or mixup/cutmix.
3. Keeps an EMA (weight-averaged) copy. After every epoch it measures the EMA copy on clean training photos, the validation set
   and the test set, and also records the running training accuracy on augmented photos.
4. Stops early when the watched value (validation accuracy or loss) hasn't improved for `PATIENCE` epochs, keeps the best epoch,
   and saves the history, the result and the test predictions.

In [ ]:
%%writefile /kaggle/working/train_clip.py
import argparse, copy, gc, json, math, os, random, time, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode

parser = argparse.ArgumentParser()
parser.add_argument("--work", required=True)           # the folder with the split and the settings (Part 3.1)
args = parser.parse_args()
with open(os.path.join(args.work, "config.json")) as f:
    C = json.load(f)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
LAYER_DECAY, HEAD_LR_MULT, GRAD_CLIP, WARMUP_EPOCHS = 0.75, 5.0, 1.0, 1

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

class Photos(Dataset):
    def __init__(self, frame, tf):
        self.files, self.labels, self.tf = frame["filename"].tolist(), frame["label_id"].tolist(), tf
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        return self.tf(Image.open(os.path.join(C["img_dir"], self.files[i])).convert("RGB")), self.labels[i]

def make_transforms(augment, mean, std):
    size = C["img_size"]
    if augment == "strong":
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.4, 1.0), interpolation=InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(),
            transforms.TrivialAugmentWide(interpolation=InterpolationMode.BILINEAR),
            transforms.ToTensor(), transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.25, value="random")])
    else:   # the same augmentation as notebook 03
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.4, 1.0), interpolation=InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(), transforms.RandomRotation(10, interpolation=InterpolationMode.BILINEAR),
            transforms.ColorJitter(0.2, 0.2), transforms.ToTensor(), transforms.Normalize(mean, std)])
    eval_tf = transforms.Compose([
        transforms.Resize((size, size), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    return train_tf, eval_tf

def loader(part, tf, train):
    frame = pd.read_csv(os.path.join(args.work, f"{part}.csv"))
    return DataLoader(Photos(frame, tf), batch_size=C["batch_size"] if train else 128, shuffle=train, drop_last=train,
                      num_workers=C["workers"], pin_memory=use_amp, persistent_workers=C["workers"] > 0)

def layer_id(name, num_blocks):
    if name.startswith(("cls_token", "pos_embed", "patch_embed", "norm_pre")):
        return 0                                  # input embeddings: the lowest layer
    if name.startswith("blocks."):
        return int(name.split(".")[1]) + 1        # transformer blocks 1-12
    return num_blocks + 1                         # final norm and classifier: the top

def param_groups(net, base_lr, freeze_blocks, weight_decay):
    num_blocks = len(net.blocks)
    head_ids = {id(p) for p in net.get_classifier().parameters()}
    groups = {}
    for name, p in net.named_parameters():
        lid = layer_id(name, num_blocks)
        is_head = id(p) in head_ids
        if not is_head and lid <= freeze_blocks:
            p.requires_grad_(False)                   # frozen: not trained
            continue
        lr = base_lr * HEAD_LR_MULT if is_head else base_lr * LAYER_DECAY ** (num_blocks + 1 - lid)
        no_decay = p.ndim <= 1 or name in ("cls_token", "pos_embed")
        g = groups.setdefault((lid, is_head, no_decay),
                              {"params": [], "lr": lr, "weight_decay": 0.0 if no_decay else weight_decay})
        g["params"].append(p)
    return list(groups.values())

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1 + step) / (10 + step))
        for ema_v, v in zip(self.model.state_dict().values(), model.state_dict().values()):
            if ema_v.dtype.is_floating_point:
                ema_v.mul_(d).add_(v.detach(), alpha=1 - d)
            else:
                ema_v.copy_(v)

@torch.no_grad()
def predict(net, dl):
    net.eval()
    probs, labels = [], []
    for x, y in dl:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            probs.append(net(x.to(device, non_blocking=True)).float().softmax(1).cpu())
        labels.append(y)
    return torch.cat(probs), torch.cat(labels)

def measure(net, dl):
    p, y = predict(net, dl)
    return F.nll_loss(torch.log(p + 1e-9), y).item(), (p.argmax(1) == y).float().mean().item()

def claim(folder):
    # Writes a "claimed" marker so the other GPU skips this variant. O_EXCL: only one process can create it.
    os.makedirs(folder, exist_ok=True)
    try:
        os.close(os.open(os.path.join(folder, "claimed"), os.O_CREAT | os.O_EXCL))
        return True
    except FileExistsError:
        return False

def train_variant(name, s, folder):
    seed_everything(C["seed"])
    t_start = time.time()
    net = timm.create_model(C["model_name"], pretrained=C["pretrained"], num_classes=C["num_classes"],
                            drop_rate=s["dropout"], drop_path_rate=s["drop_path"]).to(device)
    train_tf, eval_tf = make_transforms(s["augment"], C["mean"], C["std"])
    train_dl = loader("train", train_tf, True)
    evals = {"train": loader("train_eval", eval_tf, False), "val": loader("val", eval_tf, False),
             "test": loader("test", eval_tf, False)}
    groups = param_groups(net, s["lr"], s["freeze_blocks"], s["weight_decay"])
    trainable_params = [p for g in groups for p in g["params"]]
    trainable = sum(p.numel() for p in trainable_params)
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))
    steps = len(train_dl)
    total, warm = C["epochs"] * steps, WARMUP_EPOCHS * steps
    def lr_factor(t):
        if t < warm:
            return t / warm
        return max(0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm))), 0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if s["mixup"]:
        mix = Mixup(mixup_alpha=0.2, cutmix_alpha=1.0, prob=0.5, switch_prob=0.5,
                    label_smoothing=s["label_smoothing"], num_classes=C["num_classes"])
        criterion = SoftTargetCrossEntropy()
    else:
        mix, criterion = None, nn.CrossEntropyLoss(label_smoothing=s["label_smoothing"])
    ema = EMA(net, C["ema_decay"])
    higher = s["monitor"] == "val_acc"
    print(f"[{name}] training {trainable / 1e6:.1f}M of {sum(p.numel() for p in net.parameters()) / 1e6:.1f}M weights",
          flush=True)
    history, best, best_epoch, best_state, bad, step = [], None, 0, None, 0, 0
    for epoch in range(1, C["epochs"] + 1):
        t0 = time.time()
        net.train()
        run_loss, run_correct, seen = 0.0, 0, 0
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            target = y
            if mix is not None:
                x, target = mix(x, y)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = net(x)
                loss = criterion(logits.float(), target)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(trainable_params, GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update(net, step)
            step += 1
            hard = target.argmax(1) if target.ndim == 2 else target   # with mixup: the label with the bigger share
            run_loss += loss.item() * len(y)
            run_correct += (logits.argmax(1) == hard).sum().item()
            seen += len(y)
        row = {"epoch": epoch, "running_train_loss": run_loss / seen, "running_train_acc": run_correct / seen}
        for part, d in evals.items():
            row[f"{part}_loss"], row[f"{part}_acc"] = measure(ema.model, d)
        history.append(row)
        value = row["val_acc"] if higher else row["val_loss"]
        improved = best is None or (value > best if higher else value < best)
        if improved:
            best, best_epoch, bad = value, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in ema.model.state_dict().items()}
        else:
            bad += 1
        print(f"[{name}] epoch {epoch:2d} | train {row['train_acc']:.1%} (running {row['running_train_acc']:.1%}) | "
              f"val {row['val_acc']:.1%} | test {row['test_acc']:.1%} | val loss {row['val_loss']:.3f} | "
              f"{time.time() - t0:.0f}s{'  <- best' if improved else ''}", flush=True)
        if bad >= C["patience"]:
            print(f"[{name}] early stopping after epoch {epoch}", flush=True)
            break
    ema.model.load_state_dict(best_state)
    h = pd.DataFrame(history)
    b = h[h["epoch"] == best_epoch].iloc[0]
    test_probs, test_labels = predict(ema.model, evals["test"])
    result = {"variant": name, **s, "trainable (M)": round(trainable / 1e6, 2), "epochs run": len(h),
              "best epoch": int(best_epoch), "train acc": float(b["train_acc"]),
              "running train acc": float(b["running_train_acc"]), "val acc": float(b["val_acc"]),
              "test acc": float(b["test_acc"]), "val loss": float(b["val_loss"]),
              "train-val gap": float(b["train_acc"] - b["val_acc"]),
              "running gap": float(b["running_train_acc"] - b["val_acc"]),
              "bumpiness (pts)": round(float(h["val_acc"].diff().abs().dropna().mean() * 100), 2) if len(h) > 1 else 0.0,
              "validation drops": int((h["val_acc"].diff() < 0).sum()),
              "minutes": round((time.time() - t_start) / 60, 1)}
    h.to_csv(os.path.join(folder, "history.csv"), index=False)
    np.save(os.path.join(folder, "test_probs.npy"), test_probs.numpy())
    np.save(os.path.join(folder, "test_labels.npy"), test_labels.numpy())
    with open(os.path.join(folder, "result.json"), "w") as f:
        json.dump(result, f)
    print(f"[{name}] DONE: best epoch {best_epoch}, val {result['val acc']:.1%}, test {result['test acc']:.1%}, "
          f"gap {result['train-val gap'] * 100:+.1f} pts, {result['minutes']} min", flush=True)

# Work queue: go through the variants in order and train each one that nobody has claimed yet.
# When a GPU finishes a variant it takes the next free one, so both GPUs stay busy until the end.
for name in C["names"]:
    folder = os.path.join(args.work, "runs", "".join(c if c.isalnum() else "_" for c in name))
    if os.path.exists(os.path.join(folder, "result.json")) or not claim(folder):
        continue                                   # finished earlier, or the other GPU has it
    s = dict(C["defaults"])
    s.update(C["variants"][name])                  # the defaults, plus what this variant changes
    try:
        train_variant(name, s, folder)
    except Exception:
        print(f"[{name}] FAILED", flush=True)
        traceback.print_exc()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Part 4 — Train the Variants
One training process per GPU. Each process takes the next variant that nobody has started, so both GPUs stay busy.
Progress is printed every minute. Full logs are saved in `/kaggle/working/search/log_*.txt`.

In [ ]:
SCRIPT = "/kaggle/working/train_clip.py"
POLL_SECONDS = 60

# Variants started but not finished last time (the session stopped) are trained again
for name in VARIANTS:
    folder = os.path.join(WORK, "runs", "".join(c if c.isalnum() else "_" for c in name))
    if os.path.exists(os.path.join(folder, "result.json")):
        print(f"{name}: finished earlier, skipped")
    elif os.path.exists(os.path.join(folder, "claimed")):
        os.remove(os.path.join(folder, "claimed"))

procs = []
for g in range(n_procs):
    env = dict(os.environ)
    if n_gpu:
        env["CUDA_VISIBLE_DEVICES"] = str(g)       # this process sees only GPU g
    log_path = os.path.join(WORK, f"log_{g}.txt")
    log = open(log_path, "a")
    p = subprocess.Popen([sys.executable, "-u", SCRIPT, "--work", WORK], stdout=log, stderr=subprocess.STDOUT, env=env)
    procs.append((p, log, log_path))
print(f"Started {n_procs} training process(es).")

shown = [0] * len(procs)
def show_new_lines():
    for g, (_, _, path) in enumerate(procs):
        with open(path) as f:
            lines = f.read().splitlines()
        for line in lines[shown[g]:]:
            if line.startswith("[") or "Error" in line or "Traceback" in line:
                print(f"GPU {g} {line}" if n_gpu else line)
        shown[g] = len(lines)

start = time.time()
try:
    while any(p.poll() is None for p, _, _ in procs):
        time.sleep(POLL_SECONDS)
        show_new_lines()
except KeyboardInterrupt:
    for p, _, _ in procs:
        p.terminate()                               # stopping the cell also stops the training
    print("Stopped the training processes.")
    raise
show_new_lines()
for p, log, path in procs:
    log.close()
    if p.returncode != 0:
        print(f"A training process ended with an error; see {path}")
print(f"Training finished in {(time.time() - start) / 60:.0f} minutes.")

# Part 5 — Results
## 5.1 Comparison table
- **Gap** = clean training accuracy − validation accuracy at the kept epoch. About 5 points or less is a good fit.
- **Running gap** uses the training accuracy measured during training on augmented photos (the way notebook 03 did).
- **Bumpiness:** average change in validation accuracy between epochs (smaller is smoother).

In [ ]:
results, histories = [], {}
for name in VARIANTS:
    folder = os.path.join(WORK, "runs", "".join(c if c.isalnum() else "_" for c in name))
    if os.path.exists(os.path.join(folder, "result.json")):
        with open(os.path.join(folder, "result.json")) as f:
            results.append(json.load(f))
        histories[name] = pd.read_csv(os.path.join(folder, "history.csv"))
    else:
        print(f"{name}: no result (see the logs in Part 4)")
assert results, "No variant finished. Read the error lines printed in Part 4."
res = pd.DataFrame(results).set_index("variant")
cols = ["trainable (M)", "epochs run", "best epoch", "train acc", "running train acc", "val acc", "test acc",
        "train-val gap", "running gap", "bumpiness (pts)", "minutes"]
fmt = {c: "{:.1%}" for c in ("train acc", "running train acc", "val acc", "test acc")}
fmt.update({"train-val gap": "{:+.1%}", "running gap": "{:+.1%}", "bumpiness (pts)": "{:.2f}",
            "trainable (M)": "{:.2f}", "minutes": "{:.0f}"})
display(res[cols].style.format(fmt))
res.to_csv(os.path.join(OUTPUT_DIR, "clip_variants.csv"))

## 5.2 Curves for every variant
All panels share the **same y-axis**, so the curves can be compared directly. The dashed line is the kept epoch.
The thin dotted line is training accuracy measured during training on augmented photos (Part 6 explains it).

In [ ]:
from matplotlib.ticker import MaxNLocator

def curve_grid(metric, title):
    names_done = list(histories)
    ncols = 5
    nrows = math.ceil(len(names_done) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 3.7 * nrows), sharey=True, squeeze=False)
    scale = 100 if metric == "acc" else 1
    for ax, name in zip(axes.flat, names_done):
        h, r = histories[name], res.loc[name]
        for col, style, lab in ((f"train_{metric}", "o-", "train (clean)"), (f"val_{metric}", "s-", "validation"),
                                (f"test_{metric}", "^--", "test")):
            ax.plot(h["epoch"], h[col] * scale, style, ms=3, label=lab)
        if metric == "acc":
            ax.plot(h["epoch"], h["running_train_acc"] * 100, ":", color="gray", linewidth=1.2,
                    label="train (augmented, during training)")
        ax.axvline(r["best epoch"], color="gray", linestyle="--", linewidth=1)
        ax.set_title(f"{name}\ngap {r['train-val gap'] * 100:+.1f} pts, val {r['val acc']:.1%}", fontsize=9)
        ax.set_xlabel("Epoch")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(alpha=0.3)
    for ax in axes.flat[len(names_done):]:
        ax.axis("off")
    axes[0, 0].legend(fontsize=8)
    axes[0, 0].set_ylabel("Accuracy (%)" if metric == "acc" else "Loss")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

curve_grid("acc", "Accuracy: train (clean photos), validation and test after every epoch")
curve_grid("loss", "Loss: train (clean photos), validation and test after every epoch")

fig, ax = plt.subplots(figsize=(11, 5))
for name, h in histories.items():
    ax.plot(h["epoch"], h["val_acc"] * 100, "o-", ms=3, label=name)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation accuracy (%)"); ax.grid(alpha=0.3)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(fontsize=8, ncol=2); ax.set_title("Validation accuracy of all variants")
plt.tight_layout(); plt.show()

## 5.3 Choosing the variant
Rule: keep variants with a gap of **5 points or less**, then take the highest **validation** accuracy.
The test set is only reported. The second list shows all variants sorted by gap, if you prefer to choose by eye.

In [ ]:
GOOD_FIT_GAP = 0.05
eligible = res[res["train-val gap"] <= GOOD_FIT_GAP]
if len(eligible):
    chosen = eligible["val acc"].idxmax()
    print(f"CHOSEN: {chosen}")
else:
    chosen = res["val acc"].idxmax()
    print(f"No variant has a gap of 5 points or less. Highest validation accuracy: {chosen}")
r = res.loc[chosen]
print(f"  validation {r['val acc']:.1%}, test {r['test acc']:.1%}, gap {r['train-val gap'] * 100:+.1f} pts, "
      f"best epoch {r['best epoch']} of {r['epochs run']}")
print()
print("All variants, smallest gap first:")
for name, row in res.sort_values("train-val gap").iterrows():
    print(f"  {name:<34} gap {row['train-val gap'] * 100:+5.1f} pts   val {row['val acc']:.1%}   test {row['test acc']:.1%}")

# To look at another variant in Parts 6-7, write its name here, for example:
# chosen = "H: mixup / cutmix"

# Part 6 — Curves of the Chosen Variant
Training accuracy is measured two ways:
- **Clean photos, after each epoch** (solid): strict; it shows memorising, so the gap to validation is bigger.
- **Augmented photos, during training** (dashed): how notebook 03 measured it. Augmentation and dropout make the photos
  harder, so this line is lower and the gap looks smaller.

Both are correct; they measure different things. When showing a graph, say which one it uses.

In [ ]:
h = histories[chosen]
best_epoch = res.loc[chosen, "best epoch"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, scale in ((ax1, "acc", 100), (ax2, "loss", 1)):
    ax.plot(h["epoch"], h[f"train_{metric}"] * scale, "o-", color="C0", ms=4, label="train, clean photos (after each epoch)")
    ax.plot(h["epoch"], h[f"running_train_{metric}"] * scale, "o--", color="gray", ms=4,
            label="train, augmented photos (during training)")
    ax.plot(h["epoch"], h[f"val_{metric}"] * scale, "s-", color="C1", ms=4, label="validation")
    ax.axvline(best_epoch, color="gray", linestyle="--", linewidth=1, label=f"kept epoch ({best_epoch})")
    ax.set_xlabel("Epoch"); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.3); ax.legend()
ax1.set_ylabel("Accuracy (%)"); ax1.set_title(f"{chosen}: accuracy")
ax2.set_ylabel("Loss"); ax2.set_title(f"{chosen}: loss")
plt.tight_layout(); plt.show()

# Part 7 — The Chosen Variant on the Test Set

In [ ]:
folder = os.path.join(WORK, "runs", "".join(c if c.isalnum() else "_" for c in chosen))
test_probs = np.load(os.path.join(folder, "test_probs.npy"))
y_true = np.load(os.path.join(folder, "test_labels.npy"))
y_pred = test_probs.argmax(1)
print(f"{chosen}: test accuracy {(y_pred == y_true).mean():.1%}")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(class_names, rotation=60, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(class_names)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Confusion matrix: {chosen} (test set)")
fig.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

pairs = sorted(((cm[i, j], i, j) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j), reverse=True)[:5]
print("Most common mix-ups:")
for n, i, j in pairs:
    print(f"  {class_names[i]:>20} predicted as {class_names[j]:<20} {n} times")

# Summary
- **Ten CLIP ViT-B/16 variants**, each changing one setting from the best run so far (Part 0 table), trained with the same
  code and the same split.
- **Curves** for every variant on a shared scale (Part 5.2), with both ways of measuring training accuracy (Part 6).
- **The chosen variant** comes from a written rule: gap of 5 points or less, then the highest validation accuracy.
  The test set is reported only.